# 05 — `__name__` and the `__main__` Pattern

## Learning objectives

By the end of this notebook, you will understand what `__name__` means, how direct execution differs from importing, how the main guard works, why top-level statements run during imports, and how Python caches imported modules.

> **Important:** Examples that involve multiple `.py` files are shown as code snippets. Run each snippet in files with the indicated names when you want to observe the exact import behavior. The code cells in this notebook are self-contained demonstrations where possible.

## 01. Understanding `__name__`

Every Python module has a special module-level variable named `__name__`. Python assigns its value automatically.

- When a file is executed as the main program, its value is `"__main__"`.
- When a file is imported as a module, its value is usually the module's import name, such as `"demo"`.

It is different from an ordinary variable because Python sets it as part of the module's execution environment. You can read it, but normally you should not assign to it yourself.

In [ ]:
print(__name__)

**Expected behavior:** In a notebook, the value is typically a notebook-specific name rather than `"__main__"`. If you put `print(__name__)` in a file named `demo.py` and run `python demo.py` from a terminal, it prints:

```text
__main__
```

## 02. Direct execution versus importing

### Direct execution

When you run a file directly, Python treats that file as the program's entry point and sets its `__name__` to `"__main__"`.

```python
# demo.py
print("Module is loading")
print("Module name:", __name__)
```

Run `python demo.py`:

```text
Module is loading
Module name: __main__
```

### Importing from another file

If `main.py` imports `demo.py`, Python executes the top-level statements in `demo.py` the first time it is imported. In that context, `demo.__name__` is `"demo"`.

```python
# main.py
import demo
print("Main program is running")
```

If `demo.py` contains the two print statements above, running `python main.py` produces:

```text
Module is loading
Module name: demo
Main program is running
```

The order matters: the imported module is initialized before execution continues to the next statement in `main.py`.

In [ ]:
# A small notebook-safe illustration of the current module's name
print("Current module name:", __name__)

## 03. The main guard

The main guard is a conditional statement that runs a block only when the file is executed as the main program.

```python
if __name__ == "__main__":
    # code for direct execution goes here
```

Use `==` to compare values. Do not use `=` here, because `=` is assignment syntax, not comparison syntax.

A reusable module should usually define its functions and classes outside the guard. The guard can contain a demonstration, command-line entry point, or other code that should not automatically run just because another module imports it.

### Before and after

Without a main guard:

```python
# calculator.py
def add(a, b):
    return a + b

print("Calculator demonstration")
print(add(10, 20))
```

Importing `calculator` also prints the demonstration.

With a main guard:

```python
# calculator.py
def add(a, b):
    return a + b

if __name__ == "__main__":
    print("Calculator demonstration")
    print(add(10, 20))
```

Now `from calculator import add` imports the reusable function without running the demonstration. Running `python calculator.py` directly still runs it.

In [ ]:
def add(a, b):
    return a + b

# This cell is a notebook illustration. The main-guard check is deliberately
# shown as a value rather than wrapping notebook execution logic in a guard.
print("add(5, 7) =", add(5, 7))
print('Main-guard condition in this environment:', __name__ == "__main__")

## 04. Module initialization and import caching

When Python first imports a module, it executes the module's top-level statements. Function and class bodies are generally not executed at definition time; their definitions are created. Any other top-level statements, such as `print(...)` or calculations, do execute.

Python normally stores imported modules in `sys.modules`. A later `import` of the same module in the same interpreter usually reuses the cached module instead of executing its top-level statements again.

```python
# demo.py
print("demo is initializing")
```

```python
# main.py
import demo
import demo
print("Finished")
```

Assuming `demo` has not already been imported in that interpreter, the output is normally:

```text
demo is initializing
Finished
```

The second `import demo` does not normally print the initialization message again.

In [ ]:
import sys
print("Is 'sys' already cached?", "sys" in sys.modules)
print("Number of cached modules:", len(sys.modules))

For development, `importlib.reload(module)` can request a reload. Reloading has caveats: names imported earlier with `from module import name` and existing object references are not automatically replaced everywhere. Reloading is not a substitute for understanding module dependencies.

In [ ]:
import importlib
import math

reloaded_math = importlib.reload(math)
print("Reload returned the math module:", reloaded_math is math)

## 05. Testing reusable modules and using `main()`

A clean pattern is to put the program's entry point in a `main()` function. Keep functions intended for reuse outside the main guard.

```python
def calculate_average(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return sum(marks) / len(marks)

def main():
    marks = [80, 90, 85]
    average = calculate_average(marks)
    print("Average:", average)

if __name__ == "__main__":
    main()
```

If this code is in `student_marks.py`:

- Running `python student_marks.py` calls `main()` and prints the average.
- Importing `calculate_average` from `student_marks` makes the function available without calling `main()`.

This arrangement makes reusable logic easier to import and test. A test can call `calculate_average([80, 90])` directly and check the returned value without triggering the demonstration.

In [ ]:
def calculate_average(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return sum(marks) / len(marks)

print("Average:", calculate_average([80, 90, 85]))

The code cell above demonstrates the reusable function directly in this notebook. In a `.py` module, put the demonstration or application startup under the main guard when it should run only during direct execution.

## 06. Common mistakes and good practices

| Common mistake | Better approach |
|---|---|
| `if __name__ = "__main__":` | Use `==` for comparison: `if __name__ == "__main__":` |
| Writing `if __name__ == "__main":` | Include both underscores on each side of `main`: `"__main__"` |
| Forgetting the double underscores in `__name__` | Write the special variable exactly as `__name__` |
| Putting all reusable functions inside the guard | Define reusable functions and classes outside the guard |
| Assuming importing a module runs only function definitions | Remember that top-level statements execute during the first import |
| Assuming every import reruns a module | Remember that imported modules are normally cached in `sys.modules` |
| Putting a large amount of startup code at module level | Use a `main()` function and call it from the guard |
| Treating notebook execution exactly like running a `.py` file | Test direct-execution behavior in a real Python file when needed |

## Key takeaways

- `__name__` is a special variable whose value depends on how a module is being used.
- A directly executed Python file has `__name__ == "__main__"`.
- When imported, a module normally has its module name as `__name__`.
- Top-level statements execute when a module is initialized.
- The main guard prevents demonstration or startup code from running automatically during import.
- Python normally caches imported modules in `sys.modules`.
- Keep reusable functions outside the main guard, and use `main()` as a clear entry point when appropriate.